In [19]:
"""
╔══════════════════════════════════════════════════════════════════════╗
║       📊  PORTFOLIO RISK ANALYSIS TOOL  v4.0                         ║
║  CAPM-Calibrated Forecasts  ·  Economic Conditions  ·  3D Risk       ║
║  Diversification Engine  ·  Portfolio Beta  ·  Full Analytics        ║
╚══════════════════════════════════════════════════════════════════════╝

Install:  pip install yfinance numpy pandas matplotlib seaborn scipy
Run:      python portfolio_analyzer.py
"""

# ─────────────────────────────────────────────────────────────────────
# IMPORTS
# ─────────────────────────────────────────────────────────────────────
import sys, warnings, textwrap
from datetime import datetime, timedelta

import numpy  as np
import pandas as pd
import matplotlib; matplotlib.use("TkAgg" if sys.platform == "win32" else "Agg" if not sys.stdout.isatty() else matplotlib.get_backend())
import matplotlib.pyplot    as plt
import matplotlib.gridspec  as gridspec
import matplotlib.patches   as mpatches
from   mpl_toolkits.mplot3d import Axes3D          # noqa: F401
import seaborn as sns
import yfinance as yf
from   scipy.optimize import minimize

warnings.filterwarnings("ignore")

# ─────────────────────────────────────────────────────────────────────
# GLOBAL CONFIG
# ─────────────────────────────────────────────────────────────────────
RF            = 0.043     # Risk-free rate (US T-Bill proxy)
MKT_TICKER    = "^GSPC"
LOOKBACK      = 3         # Historical window (years)
TRADING_DAYS  = 252
ERP           = 0.055     # Long-run Equity Risk Premium (5.5%) for CAPM projections
INFLATION_RT  = 0.030     # Assumed inflation for benchmark
N_MC_CLOUD    = 8_000     # Portfolios for efficient-frontier cloud
N_MC_PROJ     = 2_000     # Forward simulation paths
PROJ_YEARS    = 10
MIN_W         = 0.02      # Per-stock floor
MAX_W         = 0.40      # Per-stock cap (keeps diversification)

# ─────────────────────────────────────────────────────────────────────
# COLOUR PALETTE
# ─────────────────────────────────────────────────────────────────────
BG   = "#0d1117"; CARD = "#161b22"; ACC  = "#58a6ff"
POS  = "#3fb950"; NEG  = "#f85149"; WARN = "#e3b341"
TXT  = "#e6edf3"; MUT  = "#8b949e"; USR  = "#c678dd"; RP   = "#56b6c2"

# ─────────────────────────────────────────────────────────────────────
# SECTOR & DIVERSIFICATION DATABASE
# ─────────────────────────────────────────────────────────────────────
SECTOR_MAP = {
    "AAPL":"Technology","MSFT":"Technology","GOOGL":"Technology","GOOG":"Technology",
    "NVDA":"Technology","AMD":"Technology","META":"Technology","AMZN":"Technology",
    "ORCL":"Technology","CRM":"Technology","ADBE":"Technology","INTC":"Technology",
    "TSLA":"Consumer Discretionary","HD":"Consumer Discretionary","NKE":"Consumer Discretionary",
    "MCD":"Consumer Discretionary","SBUX":"Consumer Discretionary","TGT":"Consumer Discretionary",
    "JPM":"Financials","BAC":"Financials","GS":"Financials","MS":"Financials",
    "WFC":"Financials","C":"Financials","BRK-B":"Financials","V":"Financials","MA":"Financials",
    "JNJ":"Healthcare","UNH":"Healthcare","LLY":"Healthcare","ABT":"Healthcare",
    "MRK":"Healthcare","PFE":"Healthcare","TMO":"Healthcare","ISRG":"Healthcare",
    "XOM":"Energy","CVX":"Energy","COP":"Energy","SLB":"Energy","EOG":"Energy",
    "HON":"Industrials","CAT":"Industrials","DE":"Industrials","BA":"Industrials",
    "GE":"Industrials","LMT":"Industrials","RTX":"Industrials",
    "WMT":"Consumer Staples","KO":"Consumer Staples","PG":"Consumer Staples",
    "COST":"Consumer Staples","PEP":"Consumer Staples","MO":"Consumer Staples",
    "NEE":"Utilities","DUK":"Utilities","SO":"Utilities","D":"Utilities",
    "LIN":"Materials","APD":"Materials","NEM":"Materials","FCX":"Materials",
    "AMT":"Real Estate","PLD":"Real Estate","SPG":"Real Estate","EQIX":"Real Estate",
    "T":"Communication Services","VZ":"Communication Services","CMCSA":"Communication Services",
    "SPY":"US Broad ETF","QQQ":"Technology ETF","IWM":"Small Cap ETF",
    "TLT":"Long Bonds","IEF":"Med Bonds","AGG":"Broad Bonds","BND":"Broad Bonds",
    "LQD":"Corp Bonds","HYG":"High Yield Bonds",
    "GLD":"Gold","SLV":"Silver","GDX":"Gold Miners","USO":"Oil",
    "DBC":"Broad Commodities","PDBC":"Broad Commodities",
    "VEA":"Intl Developed","EFA":"Intl Developed","IEFA":"Intl Developed",
    "EEM":"Emerging Markets","VWO":"Emerging Markets",
    "VNQ":"REITs","XLRE":"REITs",
}

SECTOR_DIVERSIFIERS = {
    "Technology": {
        "recs": [("MSFT","Software/cloud leader with diversified AI revenue"),
                 ("GOOGL","Search monopoly + cloud growth + cheapest Mag-7 by PE"),
                 ("AMD", "NVDA competitor benefiting from same AI wave, lower valuation")],
        "why":  "Adds high-growth, innovation exposure; historically outperforms in expansion phases.",
    },
    "Healthcare": {
        "recs": [("UNH","Largest US health insurer with consistent 15%+ EPS growth"),
                 ("LLY","GLP-1 weight-loss drugs driving decade of runway growth"),
                 ("ISRG","Surgical robotics monopoly; recession-resistant procedure demand")],
        "why":  "Defensive sector with inelastic demand. Low ~0.3 correlation to tech. Aging demographics tailwind.",
    },
    "Financials": {
        "recs": [("JPM","Best-in-class bank; benefits from higher-for-longer rate environment"),
                 ("V",  "Payments network; inflation pass-through built-in; asset-light model"),
                 ("BRK-B","Buffett's conglomerate; built-in diversification + value discipline")],
        "why":  "Banks benefit from elevated interest rates. Value-tilt offsets growth-heavy portfolios.",
    },
    "Energy": {
        "recs": [("XOM","Integrated oil major; strong free cash flow even at $60/barrel"),
                 ("CVX","Lower break-even than peers; 4%+ dividend; geopolitical hedge"),
                 ("COP","Pure-play E&P with disciplined capex and shareholder returns")],
        "why":  "Inflation hedge. Low/negative correlation to tech. Commodity supercycle exposure.",
    },
    "Consumer Staples": {
        "recs": [("KO", "170-year brand moat; pricing power in inflationary environment"),
                 ("WMT","Recession-proof; gaining market share across income levels"),
                 ("PG", "75+ years of dividend growth; global household brand portfolio")],
        "why":  "Maximum defensive quality. Outperforms sharply in recessions. Dividend income.",
    },
    "Industrials": {
        "recs": [("CAT","Infrastructure + commodity supercycle; AI-data-centre construction"),
                 ("HON","Diversified industrial conglomerate; aerospace + building automation"),
                 ("DE", "Agricultural equipment monopoly; food security megatrend")],
        "why":  "Re-shoring + infrastructure spending tailwind. Cyclical complement to defensives.",
    },
    "Utilities": {
        "recs": [("NEE","Largest US utility; 60%+ renewable; AI data-centre power demand"),
                 ("DUK","Regulated utility; rate-increase certainty; 4%+ yield"),
                 ("SO", "Regulated monopoly in high-growth Sun Belt markets")],
        "why":  "Bond proxy; inversely correlated to rate rises but benefits from AI electricity demand.",
    },
    "Long Bonds": {
        "recs": [("TLT","20-year Treasury ETF; rallies when economy weakens or rates fall"),
                 ("IEF","7-10yr Treasury; less rate-sensitive than TLT; liquid hedge"),
                 ("AGG","Broad US bond market; Sharpe diversifier for equity-heavy portfolios")],
        "why":  "Negative or near-zero correlation to equities in crises. Capital preservation.",
    },
    "Gold": {
        "recs": [("GLD","Physical gold ETF; ultimate tail-risk hedge; dollar hedge"),
                 ("GDX","Gold miners ETF; leveraged to gold price; higher vol + return"),
                 ("IAU","iShares gold ETF; lower expense ratio alternative to GLD")],
        "why":  "Crisis hedge + USD hedge. Historically outperforms when real rates are negative.",
    },
    "Intl Developed": {
        "recs": [("VEA","Vanguard developed markets ex-US; Europe + Japan + Canada"),
                 ("EFA","iShares developed markets; geographic concentration reducer"),
                 ("IEFA","Core international; lower cost; broad developed exposure")],
        "why":  "USD depreciation hedge. Valuation cheaper than US. Geopolitical diversification.",
    },
    "Emerging Markets": {
        "recs": [("EEM","Emerging markets ETF; China + India + Brazil + Taiwan"),
                 ("VWO","Vanguard EM; slightly different weights; lower cost"),
                 ("IEMG","iShares Core EM; largest EM ETF by AUM")],
        "why":  "Higher long-run growth potential. Adds currency diversification. Mean-reversion candidate.",
    },
    "REITs": {
        "recs": [("AMT","Cell-tower REIT; digital infrastructure; contractual rent escalators"),
                 ("PLD","Industrial REIT; e-commerce warehouses; secular demand growth"),
                 ("EQIX","Data centre REIT; AI infrastructure play; global monopoly positioning")],
        "why":  "Inflation-linked income. Low stock correlation historically. Real asset exposure.",
    },
}

# ─────────────────────────────────────────────────────────────────────
# SECTION 1  ·  USER INPUT
# ─────────────────────────────────────────────────────────────────────
BANNER = """
╔══════════════════════════════════════════════════════════════════════╗
║       📊  PORTFOLIO RISK ANALYSIS TOOL  v4.0                         ║
║  CAPM-Calibrated Forecasts  ·  Economic Conditions  ·  3D Risk       ║
╚══════════════════════════════════════════════════════════════════════╝"""

def _flt(s, default=0.0):
    try:    return float(str(s).strip().replace(",","").replace("$","").replace("%",""))
    except: return default

def get_user_inputs():
    print(BANNER)
    sep = "  " + "─"*62

    print(f"\n{sep}")
    print("  STEP 1/4  ·  Tickers  (e.g.  AAPL, MSFT, NVDA, JPM, GLD)")
    print(f"{sep}")
    while True:
        raw = input("\n  ▶ Tickers: ").strip()
        tickers = [t.strip().upper() for t in raw.split(",") if t.strip()]
        if len(tickers) >= 2: break
        print("  ⚠️  Enter at least 2 tickers.")

    print(f"\n{sep}")
    print("  STEP 2/4  ·  Weighting Mode")
    print(f"{sep}")
    print("  1  →  Optimal only  (Max Sharpe / Min Vol / Risk Parity)")
    print("  2  →  Manual only   (you specify allocations + cash)")
    print("  3  →  Both          (compare yours vs optimal)  ★ recommended")
    while True:
        m = input("\n  ▶ [1/2/3]: ").strip()
        if m in ("1","2","3"): mode = int(m); break
        print("  ⚠️  Enter 1, 2, or 3.")

    user_sw, user_cw = None, 0.0
    if mode in (2,3):
        print(f"\n{sep}")
        print("  STEP 3/4  ·  Your Weights  (enter as %, e.g. 25 for 25%)")
        print(f"{sep}\n")
        raw_w = {}
        for t in tickers:
            raw_w[t] = max(0.0, _flt(input(f"    {t:>8}  (%): "), 0.0))
        cash_pct = max(0.0, _flt(input(f"    {'CASH':>8}  (%): "), 0.0))

        st = sum(raw_w.values()); grand = st + cash_pct
        if grand == 0:
            raw_w = {t: 100/len(tickers) for t in tickers}; st = 100; grand = 100
        if grand > 100:
            f = 100/grand; raw_w = {t:v*f for t,v in raw_w.items()}; cash_pct *= f; st *= f
            print("  ⚠️  Normalised to 100%.")

        user_cw = cash_pct/100
        rem = 1.0 - user_cw
        user_sw = np.array([raw_w[t]/100/st*rem for t in tickers]) if st > 0 \
                  else np.ones(len(tickers))*rem/len(tickers)
        print(f"\n  ✅  Weights: {', '.join(f'{t}={w*100:.1f}%' for t,w in zip(tickers,user_sw))}  Cash={user_cw*100:.1f}%")
    else:
        print(f"\n{sep}\n  STEP 3/4  ·  Skipped (auto mode)\n{sep}")

    print(f"\n{sep}")
    print("  STEP 4/4  ·  Initial Investment Amount")
    print(f"{sep}")
    initial_value = max(100.0, _flt(input("\n  ▶ Starting value ($, Enter=10000): "), 10_000.0))
    print(f"\n  ✅  Starting value: ${initial_value:,.2f}\n")
    return tickers, mode, user_sw, user_cw, initial_value

# ─────────────────────────────────────────────────────────────────────
# SECTION 2  ·  DATA FETCHING
# ─────────────────────────────────────────────────────────────────────

def _extract_close(raw, syms):
    """Robustly extract closing prices from any yfinance version."""
    if isinstance(raw.columns, pd.MultiIndex):
        lvl0 = raw.columns.get_level_values(0).unique().tolist()
        field = "Adj Close" if "Adj Close" in lvl0 else "Close"
        return raw[field].copy()
    else:
        field = "Adj Close" if "Adj Close" in raw.columns else "Close"
        df = raw[[field]].copy()
        if len(syms) == 1: df.columns = syms
        return df

def fetch_prices(tickers):
    end = datetime.today(); start = end - timedelta(days=LOOKBACK*365+30)
    syms = list(dict.fromkeys(tickers + [MKT_TICKER]))
    print(f"  ⏳  Downloading {LOOKBACK}yr price data for {len(syms)} symbols …")
    raw = yf.download(syms, start=start, end=end, auto_adjust=True, progress=False)
    prices = _extract_close(raw, syms).ffill().bfill().dropna(how="all")
    market = prices[MKT_TICKER].copy() if MKT_TICKER in prices.columns else None
    stocks = prices[[t for t in tickers if t in prices.columns]].copy()
    return stocks, market

def validate(tickers, prices):
    valid = [t for t in tickers if t in prices.columns and prices[t].notna().sum()>60]
    bad   = [t for t in tickers if t not in valid]
    if bad: print(f"  ⚠️  Skipping: {', '.join(bad)}")
    return valid

def fetch_economic_data():
    """Fetch economic indicators; returns partial dict on failure."""
    symbols = {
        "^VIX":"VIX",  "^TNX":"10Y_YIELD",  "^IRX":"3M_YIELD",
        "^TYX":"30Y_YIELD","^GSPC":"SPX",
        "XLK":"XLK","XLF":"XLF","XLV":"XLV","XLE":"XLE",
        "XLI":"XLI","XLU":"XLU","XLB":"XLB","XLRE":"XLRE",
        "GLD":"GOLD","TLT":"TLT",
    }
    eco = {}
    print("  📡  Fetching economic indicators …")
    for sym, key in symbols.items():
        try:
            raw = yf.download(sym, period="1y", progress=False)
            if len(raw) < 5: continue
            cl = _extract_close(raw,[sym]).iloc[:,0].dropna()
            if len(cl) < 5: continue
            eco[key] = {
                "sym":     sym,
                "series":  cl,
                "current": float(cl.iloc[-1]),
                "prev1m":  float(cl.iloc[-22]) if len(cl)>22 else float(cl.iloc[0]),
                "chg1m":   float(cl.iloc[-1]/cl.iloc[-22]-1) if len(cl)>22 else 0.0,
                "chg1y":   float(cl.iloc[-1]/cl.iloc[0]-1),
            }
        except Exception:
            pass
    if eco: print(f"  ✅  Economic data: {len(eco)} indicators loaded")
    else:   print("  ⚠️  Economic data unavailable (no internet / timeout)")
    return eco

# ─────────────────────────────────────────────────────────────────────
# SECTION 3  ·  METRICS
# ─────────────────────────────────────────────────────────────────────

def ann_ret(r):  return (1+r.mean())**TRADING_DAYS - 1
def ann_vol(r):  return r.std() * np.sqrt(TRADING_DAYS)
def sharpe(r,v): return (r-RF)/v if v>0 else 0.0

def sortino_s(returns):
    ar = ann_ret(returns); dv = returns.clip(upper=0).std()*np.sqrt(TRADING_DAYS)
    return (ar-RF)/dv if dv>0 else 0.0

def max_dd(prices):
    return float(((prices-prices.cummax())/prices.cummax()).min())

def calmar_s(returns, prices):
    mdd = abs(max_dd(prices)); return (ann_ret(returns)/mdd) if mdd>0 else 0.0

def hist_var(s, c=0.95):  return float(-np.percentile(s.dropna(),(1-c)*100))
def hist_cvar(s, c=0.95):
    v=hist_var(s,c); t=s[s<=-v]; return float(-t.mean()) if len(t) else v

def compute_betas(returns, mkt):
    betas = {}
    mkt = mkt.dropna()
    for col in returns.columns:
        s = returns[col].dropna(); idx = s.index.intersection(mkt.index)
        if len(idx)<30: betas[col]=np.nan; continue
        cm = np.cov(s.loc[idx].values, mkt.loc[idx].values)
        betas[col] = float(cm[0,1]/mkt.loc[idx].var()) if mkt.loc[idx].var()>0 else np.nan
    return betas

def capm_forward(beta): return RF + beta*ERP

def classify_risk(v):
    if v<0.10: return "LOW",      POS
    if v<0.18: return "MODERATE", ACC
    if v<0.28: return "HIGH",     WARN
    return          "VERY HIGH",  NEG

def classify_beta(b):
    if np.isnan(b): return "N/A"
    if b<0:   return "Inverse"
    if b<0.5: return "Defensive"
    if b<0.85:return "Low Sensitivity"
    if b<1.15:return "Market-Like"
    if b<1.5: return "Aggressive"
    return          "Very Aggressive"

# ─────────────────────────────────────────────────────────────────────
# SECTION 4  ·  PORTFOLIO OPTIMISATION
# ─────────────────────────────────────────────────────────────────────

def port_metrics(sw, cw, fwd_ret, cov):
    """fwd_ret = CAPM-based forward returns (not historical)."""
    r  = float(sw@fwd_ret) + cw*RF
    v  = float(np.sqrt(sw@cov@sw))
    sr = sharpe(r,v)
    return r, v, sr

def port_beta(sw, bv): return float(sw@bv)

def _auto_cash(cov, w):
    v = float(np.sqrt(w@cov@w))
    if v<0.10: return 0.05
    if v<0.15: return 0.08
    if v<0.20: return 0.12
    if v<0.25: return 0.18
    return 0.25

def apply_cash(w, pct): return w*(1-pct), pct

def opt_weights(fwd_ret, cov, obj="sharpe"):
    n   = len(fwd_ret); w0 = np.ones(n)/n
    bds = [(MIN_W,MAX_W)]*n
    con = [{"type":"eq","fun":lambda w: w.sum()-1}]
    fns = {
        "sharpe":  lambda w: -(w@fwd_ret-RF)/max(np.sqrt(w@cov@w),1e-9),
        "min_vol": lambda w: np.sqrt(w@cov@w),
    }
    res = minimize(fns[obj], w0, method="SLSQP", bounds=bds, constraints=con,
                   options={"maxiter":2000,"ftol":1e-14})
    return res.x if res.success else w0

def risk_parity(cov):
    n=cov.shape[0]; w0=np.ones(n)/n
    def obj(w):
        w=np.maximum(w,1e-9); pv=np.sqrt(w@cov@w)
        if pv<1e-12: return 1e10
        rc=w*(cov@w/pv); return float(np.sum((rc-pv/n)**2))
    res=minimize(obj,w0,method="SLSQP",bounds=[(1e-4,1)]*n,
                 constraints=[{"type":"eq","fun":lambda w:w.sum()-1}],
                 options={"maxiter":3000,"ftol":1e-14})
    return np.maximum(res.x,0) if res.success else w0

def mc_cloud(fwd_ret, cov, n=N_MC_CLOUD):
    k=len(fwd_ret); rng=np.random.default_rng(7)
    ws=rng.dirichlet(np.ones(k),size=n)
    rets=ws@fwd_ret; vols=np.sqrt(np.einsum("ni,ij,nj->n",ws,cov,ws))
    srs=(rets-RF)/np.where(vols>0,vols,1e-9)
    return np.vstack([vols,rets,srs])

# ─────────────────────────────────────────────────────────────────────
# SECTION 5  ·  CAPM-CALIBRATED BLOCK-BOOTSTRAP SIMULATION
#
# WHY BOOTSTRAP INSTEAD OF PARAMETRIC GBM?
# - Parametric GBM with historical returns over-inflates long-run forecasts
#   when historical periods contain bubbles (e.g. NVDA 2022-2025).
# - Block bootstrap resamples actual daily returns (preserving fat-tails
#   and serial correlation) but scales the DRIFT to match the CAPM expected
#   return, keeping the SPREAD (volatility) realistic.
# ─────────────────────────────────────────────────────────────────────

def simulate_capm_bootstrap(sw, cw, daily_returns_df, fwd_ret, cov,
                             n_years=PROJ_YEARS, n_paths=N_MC_PROJ,
                             initial=10_000.0, seed=42, block=20):
    """
    Block bootstrap with CAPM drift correction.
    sw        : stock weights (sum = 1-cw)
    cw        : cash fraction
    fwd_ret   : CAPM-based annual expected returns per stock
    cov       : annual covariance matrix
    block     : bootstrap block length (trading days, ~1 month)
    """
    rng       = np.random.default_rng(seed)
    n_steps   = n_years * TRADING_DAYS

    # ── Step 1: compute portfolio DAILY returns from historical data ──
    port_daily = (daily_returns_df @ sw).values   # shape (T,)
    T          = len(port_daily)

    # ── Step 2: CAPM annual drift & historical vol ───────────────────
    capm_ann   = float(sw @ fwd_ret) + cw * RF
    hist_ann   = float((1 + (daily_returns_df @ sw).mean()) ** TRADING_DAYS - 1)
    port_vol   = float(np.sqrt(sw @ cov @ sw))   # annualised historical vol

    # Daily adjustment: scale each bootstrap block so its expected
    # drift matches CAPM rather than historical (key calibration step)
    hist_daily  = hist_ann / TRADING_DAYS
    capm_daily  = capm_ann / TRADING_DAYS
    drift_adj   = capm_daily - hist_daily    # daily drift correction

    paths = np.empty((n_paths, n_steps + 1))
    paths[:, 0] = initial

    for p in range(n_paths):
        val = initial
        for step in range(0, n_steps, block):
            # Random block start (circular)
            start  = rng.integers(0, max(1, T - block))
            blk    = port_daily[start:start+block]
            actual = min(block, n_steps - step)
            # Apply drift correction to each day in the block
            adj_blk = blk[:actual] + drift_adj
            for d in range(actual):
                val *= (1.0 + adj_blk[d])
                paths[p, step + d + 1] = val

    # Cash grows deterministically at RF
    if cw > 0:
        cash_growth = np.exp(RF/TRADING_DAYS * np.arange(n_steps+1))
        stock_frac  = 1.0 - cw
        paths = paths * stock_frac + initial * cw * cash_growth[None,:]

    return paths

# ─────────────────────────────────────────────────────────────────────
# SECTION 6  ·  ECONOMIC REGIME ANALYSIS
# ─────────────────────────────────────────────────────────────────────

def analyse_regime(eco):
    """Return dict with regime classification and narrative."""
    regime = {"label":"UNKNOWN","color":MUT,"bullets":[]}

    vix = eco.get("VIX",{}).get("current", None)
    y10 = eco.get("10Y_YIELD",{}).get("current", None)
    y3m = eco.get("3M_YIELD",{}).get("current", None)
    spx_chg = eco.get("SPX",{}).get("chg1y", None)

    # VIX assessment
    if vix is not None:
        if   vix < 15:  regime["bullets"].append(f"VIX {vix:.1f} — CALM: Complacency risk; low fear, elevated valuations.")
        elif vix < 20:  regime["bullets"].append(f"VIX {vix:.1f} — NORMAL: Healthy risk appetite, moderate uncertainty.")
        elif vix < 30:  regime["bullets"].append(f"VIX {vix:.1f} — ELEVATED: Uncertainty rising; consider defensive tilt.")
        else:           regime["bullets"].append(f"VIX {vix:.1f} — STRESS: Market in fear; historically good entry points emerge.")

    # Yield curve
    if y10 is not None and y3m is not None:
        spread = y10 - y3m
        if spread < -0.005:
            regime["bullets"].append(f"Yield curve INVERTED ({spread:+.2f}pp): Recession predictor; favour defensives.")
            regime["label"] = "LATE CYCLE / RECESSION RISK"; regime["color"] = NEG
        elif spread < 0.50:
            regime["bullets"].append(f"Yield curve FLAT ({spread:+.2f}pp): Late-cycle dynamics; rates may peak.")
            regime["label"] = "LATE CYCLE"; regime["color"] = WARN
        else:
            regime["bullets"].append(f"Yield curve NORMAL ({spread:+.2f}pp): Economic expansion signalled.")
            regime["label"] = "EXPANSION"; regime["color"] = POS

    # Rates level
    if y10 is not None:
        if   y10 > 5:   regime["bullets"].append(f"10yr yield {y10:.2f}% — HIGH RATES: Headwind for growth/tech & bonds.")
        elif y10 > 4:   regime["bullets"].append(f"10yr yield {y10:.2f}% — ELEVATED: Value over growth; financials benefit.")
        elif y10 > 2.5: regime["bullets"].append(f"10yr yield {y10:.2f}% — MODERATE: Balanced environment.")
        else:           regime["bullets"].append(f"10yr yield {y10:.2f}% — LOW RATES: Growth/tech favoured; bond yields unattractive.")

    # Market momentum
    if spx_chg is not None:
        if spx_chg > 0.20:  regime["bullets"].append(f"S&P 500 +{spx_chg*100:.0f}% (1yr) — BULL: Momentum strong but stretched valuations.")
        elif spx_chg > 0:   regime["bullets"].append(f"S&P 500 +{spx_chg*100:.0f}% (1yr) — POSITIVE: Moderate bull market.")
        elif spx_chg > -0.10: regime["bullets"].append(f"S&P 500 {spx_chg*100:.0f}% (1yr) — FLAT: Digesting gains; rotation underway.")
        else:               regime["bullets"].append(f"S&P 500 {spx_chg*100:.0f}% (1yr) — BEAR: Risk-off; capital preservation priority.")

    if not regime["bullets"]: regime["bullets"].append("Economic data unavailable. Configure internet access to enable live indicators.")
    return regime

# ─────────────────────────────────────────────────────────────────────
# SECTION 7  ·  DIVERSIFICATION ENGINE
# ─────────────────────────────────────────────────────────────────────

def sector_exposure(tickers, weights):
    """Return dict {sector: weight_fraction}."""
    exp = {}
    for t, w in zip(tickers, weights):
        s = SECTOR_MAP.get(t, "Other")
        exp[s] = exp.get(s, 0.0) + w
    return exp

def hhi(weights):
    """Herfindahl-Hirschman Index — 1/n=perfect, 1=monopoly."""
    return float(np.sum(np.array(weights)**2))

def diversification_score(returns):
    """Portfolio diversification ratio (weighted avg vol / portfolio vol)."""
    corr = returns.corr().values
    return float(1 - corr[np.triu_indices_from(corr,1)].mean())

def build_recommendations(tickers, sector_exp, eco_regime, corr_df):
    """
    Generate ranked diversification recommendations with full reasoning.
    Returns list of dicts.
    """
    recs = []
    present_sectors = set(sector_exp.keys())
    all_sectors     = set(SECTOR_DIVERSIFIERS.keys())
    missing_sectors = all_sectors - present_sectors

    # Priority order: defensive/decorrelated first when regime is late/recession
    if "LATE" in eco_regime.get("label","") or "RECESSION" in eco_regime.get("label",""):
        priority = ["Healthcare","Consumer Staples","Utilities","Long Bonds",
                    "Gold","Intl Developed","Emerging Markets","REITs",
                    "Financials","Industrials","Energy","Technology"]
    else:
        priority = ["Long Bonds","Gold","Intl Developed","Healthcare",
                    "Consumer Staples","Emerging Markets","REITs","Utilities",
                    "Energy","Financials","Industrials","Technology"]

    # Add missing sector recs
    for sector in priority:
        if sector in missing_sectors and sector in SECTOR_DIVERSIFIERS:
            db       = SECTOR_DIVERSIFIERS[sector]
            top_tick = db["recs"][0][0]
            top_desc = db["recs"][0][1]
            # Correlation with existing portfolio (if data available)
            avg_corr = "N/A"
            if top_tick in corr_df.columns:
                sub = corr_df[[top_tick]].loc[tickers].mean().values[0]
                avg_corr = f"{sub:.2f}"
            recs.append({
                "sector":    sector,
                "ticker":    top_tick,
                "desc":      top_desc,
                "why":       db["why"],
                "all_recs":  db["recs"],
                "avg_corr":  avg_corr,
                "type":      "MISSING SECTOR",
            })

    # Identify over-concentrated existing sectors
    for sector, w in sorted(sector_exp.items(), key=lambda x:-x[1]):
        if w > 0.40:
            recs.append({
                "sector":  sector,
                "ticker":  "— REDUCE —",
                "desc":    f"Your portfolio has {w*100:.0f}% in {sector}. Target ≤40% per sector.",
                "why":     f"Concentration risk: {w*100:.0f}% exposure means idiosyncratic shocks hit hard.",
                "all_recs": [],
                "avg_corr": "N/A",
                "type":    "OVER-CONCENTRATED",
            })

    return recs[:10]  # top-10

# ─────────────────────────────────────────────────────────────────────
# SECTION 8  ·  MATPLOTLIB STYLE
# ─────────────────────────────────────────────────────────────────────

def setup_style():
    plt.rcParams.update({
        "figure.facecolor": BG, "axes.facecolor": CARD,
        "axes.edgecolor":  "#30363d", "axes.labelcolor": MUT,
        "axes.titlecolor": TXT, "axes.titlesize":   11,
        "axes.titleweight":"bold",
        "xtick.color": MUT, "ytick.color": MUT,
        "text.color":  TXT, "grid.color":  "#21262d",
        "grid.alpha":  0.50, "font.size":  9,
        "legend.framealpha":0.35,"legend.facecolor":CARD,
        "legend.edgecolor":"#30363d","legend.fontsize":8,
    })

def _ax_dark(ax):
    ax.set_facecolor(CARD); ax.tick_params(colors=MUT)
    for sp in ax.spines.values(): sp.set_edgecolor("#30363d")

def _ax_dark3d(ax):
    ax.set_facecolor(BG)
    for pane in (ax.xaxis.pane, ax.yaxis.pane, ax.zaxis.pane):
        pane.fill = False; pane.set_edgecolor("#30363d")
    ax.tick_params(colors=MUT, labelsize=8)

# ─────────────────────────────────────────────────────────────────────
# SECTION 9  ·  FIGURE 1 — CORRELATION HEATMAP
# ─────────────────────────────────────────────────────────────────────

def fig_heatmap(returns, tickers):
    corr = returns[tickers].corr(); n = len(tickers)
    sz   = max(8, n*1.5)
    fig, ax = plt.subplots(figsize=(sz, sz*0.78), facecolor=BG)
    cmap = sns.diverging_palette(12,220,sep=10,n=256,as_cmap=True)
    sns.heatmap(corr, annot=True, fmt=".2f", cmap=cmap, vmin=-1, vmax=1,
                center=0, ax=ax, square=True, linewidths=0.6, linecolor="#30363d",
                annot_kws={"size":max(9,14-n),"weight":"bold"},
                cbar_kws={"shrink":0.72,"aspect":22,"pad":0.02})
    ax.set_title("Asset Correlation Matrix", fontsize=16, pad=18, color=TXT, fontweight="bold")
    ax.tick_params(axis="x",rotation=45,labelsize=11,colors=TXT)
    ax.tick_params(axis="y",rotation=0, labelsize=11,colors=TXT)
    cb=ax.collections[0].colorbar; cb.ax.yaxis.set_tick_params(color=MUT,labelsize=8)
    plt.setp(cb.ax.yaxis.get_ticklabels(),color=MUT)
    ax.annotate("+1=perfectly correlated  |  0=uncorrelated  |  -1=inversely correlated",
                xy=(0.5,-0.07),xycoords="axes fraction",ha="center",
                fontsize=8.5,color=MUT,style="italic")
    plt.tight_layout(pad=2.5); return fig

# ─────────────────────────────────────────────────────────────────────
# SECTION 10  ·  FIGURE 2 — MAIN DASHBOARD
# ─────────────────────────────────────────────────────────────────────

def fig_dashboard(tickers, returns, prices, betas_d, mean_ret_ann, vols_ann,
                  mc_res, cov_ann, strategies, fwd_ret_series):
    sc = plt.cm.plasma(np.linspace(0.12,0.92,len(tickers)))
    fig = plt.figure(figsize=(24,20),facecolor=BG)
    fig.suptitle("📊  Portfolio Analysis Dashboard",fontsize=22,color=TXT,fontweight="bold",y=0.99)
    gs  = gridspec.GridSpec(3,3,figure=fig,hspace=0.44,wspace=0.32,
                            left=0.055,right=0.97,top=0.95,bottom=0.05)

    # P1 — Cumulative returns
    ax1=fig.add_subplot(gs[0,:2]); cum=(1+returns[tickers]).cumprod()
    for i,t in enumerate(tickers): ax1.plot(cum.index,cum[t],label=t,lw=2,color=sc[i])
    ax1.axhline(1,color=MUT,ls="--",alpha=0.45,lw=0.9)
    ax1.set_title("Cumulative Returns (normalised to $1)"); ax1.set_ylabel("Growth of $1.00")
    ax1.legend(loc="upper left",ncol=min(len(tickers),6)); ax1.yaxis.grid(True); _ax_dark(ax1)

    # P2 — Beta bar chart
    ax2=fig.add_subplot(gs[0,2]); bv=[betas_d.get(t,0) for t in tickers]
    bars=ax2.barh(tickers,bv,color=[POS if b<=1 else NEG for b in bv],alpha=0.85,edgecolor="none",height=0.55)
    ax2.axvline(1,color=WARN,ls="--",lw=1.6,alpha=0.9,label="β=1 market"); ax2.axvline(0,color=MUT,ls="-",lw=0.7,alpha=0.4)
    for bar,val in zip(bars,bv):
        off=0.04 if val>=0 else -0.04
        ax2.text(val+off,bar.get_y()+bar.get_height()/2,f"{val:+.2f}",va="center",
                 ha="left" if val>=0 else "right",fontsize=9,color=TXT,fontweight="bold")
    ax2.set_title("Market Beta (β vs S&P 500)"); ax2.set_xlabel("Beta")
    ax2.legend(fontsize=8); ax2.xaxis.grid(True); _ax_dark(ax2)

    # P3 — Efficient frontier + MC cloud  (using CAPM forward returns)
    ax3=fig.add_subplot(gs[1,:2])
    sc3=ax3.scatter(mc_res[0]*100,mc_res[1]*100,c=mc_res[2],cmap="plasma",alpha=0.25,s=4,zorder=2)
    cb3=plt.colorbar(sc3,ax=ax3,pad=0.01); cb3.set_label("Sharpe",fontsize=8,color=MUT)
    plt.setp(cb3.ax.yaxis.get_ticklabels(),color=MUT,fontsize=8); cb3.ax.yaxis.set_tick_params(color=MUT)
    for st in strategies:
        r,v,sr=port_metrics(st["raw_w"],0,fwd_ret_series.values,cov_ann.values)
        ax3.scatter(v*100,r*100,marker=st["marker"],s=st.get("ms",180),color=st["color"],
                    zorder=6,edgecolors="white",linewidths=0.7,
                    label=f"{st['name']}  SR={sr:+.2f}")
    for i,t in enumerate(tickers):
        ax3.scatter(vols_ann[t]*100,fwd_ret_series[t]*100,
                    marker="^",s=130,color=sc[i],zorder=7,edgecolors="white",linewidths=0.8)
        ax3.annotate(t,xy=(vols_ann[t]*100,fwd_ret_series[t]*100),
                     xytext=(5,5),textcoords="offset points",fontsize=8.5,color=sc[i],fontweight="bold")
    ax3.axhline(RF*100,color=MUT,ls=":",lw=0.8,alpha=0.7)
    ax3.set_title("Efficient Frontier  ·  CAPM-Forward Returns  ·  MC Cloud")
    ax3.set_xlabel("Annual Volatility (%)"); ax3.set_ylabel("CAPM Forward Return (%)")
    ax3.legend(fontsize=8,ncol=2); ax3.grid(True,alpha=0.18); _ax_dark(ax3)

    # P4 — Risk/return bubble (historical)
    ax4=fig.add_subplot(gs[1,2])
    for i,t in enumerate(tickers):
        r=mean_ret_ann[t]*100; v=vols_ann[t]*100; sr=sharpe(mean_ret_ann[t],vols_ann[t])
        ax4.scatter(v,r,s=max(80,abs(sr)*250),color=sc[i],alpha=0.85,edgecolors="white",lw=0.8,zorder=5)
        ax4.annotate(t,xy=(v,r),xytext=(7,4),textcoords="offset points",fontsize=9,color=sc[i],fontweight="bold")
    ax4.axhline(RF*100,color=MUT,ls="--",lw=1,alpha=0.7,label=f"RF {RF*100:.1f}%")
    ax4.set_title("Risk vs Return (Historical)\n(bubble ∝ |Sharpe|)"); ax4.set_xlabel("Vol %"); ax4.set_ylabel("Hist Return %")
    ax4.legend(fontsize=8); ax4.grid(True,alpha=0.18); _ax_dark(ax4)

    # P5 — Weights + cash
    ax5=fig.add_subplot(gs[2,:2])
    xlbls=tickers+["CASH"]; xp=np.arange(len(xlbls)); ns=len(strategies)
    bw=0.78/ns; offs=np.linspace(-(ns-1)/2,(ns-1)/2,ns)*bw; maxh=0.0
    for st,off in zip(strategies,offs):
        sb=ax5.bar(xp[:-1]+off,st["sw"]*100,width=bw,label=st["name"],color=st["color"],alpha=0.88,edgecolor="none")
        ax5.bar(xp[-1]+off,st["cw"]*100,width=bw,color="#c9d1d9",alpha=0.72,edgecolor="#8b949e",lw=0.5,hatch="///")
        for rect,val in zip(sb,st["sw"]):
            h=val*100; maxh=max(maxh,h)
            if h>3.5: ax5.text(rect.get_x()+rect.get_width()/2,h+0.4,f"{h:.0f}%",ha="center",va="bottom",fontsize=7,color=TXT)
        ch=st["cw"]*100
        if ch>1: ax5.text(xp[-1]+off,ch+0.4,f"{ch:.0f}%",ha="center",va="bottom",fontsize=7,color=TXT)
    ax5.axvline(len(tickers)-0.5,color=MUT,ls="--",lw=1,alpha=0.45)
    ax5.text(len(tickers)-0.42,maxh*0.87,"CASH ▶",color=MUT,fontsize=8,style="italic")
    beta_str="  Portfolio β:  "+"   |   ".join(f"{s['name']} {s['beta']:.2f}" for s in strategies)
    ax5.annotate(beta_str,xy=(0.01,-0.14),xycoords="axes fraction",fontsize=7.5,color=MUT,style="italic")
    ax5.set_title("Recommended Weightings incl. Cash (hatched = CASH)"); ax5.set_ylabel("Allocation %")
    ax5.set_xticks(xp); ax5.set_xticklabels(xlbls,fontsize=9)
    ax5.legend(fontsize=8.5,ncol=ns); ax5.yaxis.grid(True); _ax_dark(ax5)

    # P6 — Metrics table
    ax6=fig.add_subplot(gs[2,2]); ax6.axis("off")
    hdr=["Ticker","Ret(H)","Ret(C)","Vol","Sharpe","β","Risk"]
    rows=[]
    for t in tickers:
        r=mean_ret_ann[t]; v=vols_ann[t]; beta=betas_d.get(t,np.nan); rl,_=classify_risk(v)
        rows.append([t,f"{r*100:+.1f}%",f"{fwd_ret_series[t]*100:+.1f}%",
                     f"{v*100:.1f}%",f"{sharpe(r,v):.2f}",
                     f"{beta:.2f}" if not np.isnan(beta) else "N/A",rl])
    rows.append(["─"*5]*7)
    brow=["Port β"]+[f"{s['beta']:.2f}" for s in strategies]+[""]*max(0,6-len(strategies))
    rows.append(brow[:7])
    tbl=ax6.table(cellText=rows,colLabels=hdr,loc="center",cellLoc="center")
    tbl.auto_set_font_size(False); tbl.set_fontsize(8); tbl.scale(1.05,1.75)
    nd=len(tickers)
    for (ri,ci),cell in tbl.get_celld().items():
        cell.set_edgecolor("#30363d"); cell.set_linewidth(0.5)
        if ri==0:    cell.set_facecolor(CARD); cell.set_text_props(color=ACC,fontweight="bold")
        elif ri>nd:  cell.set_facecolor(BG);   cell.set_text_props(color=WARN if ri==nd+2 else MUT,fontweight="bold" if ri==nd+2 else "normal")
        else:
            cell.set_facecolor(CARD if ri%2==0 else BG)
            txt=rows[ri-1][ci]; col=TXT
            if ci==1: col=POS if "+" in txt else NEG
            elif ci==6: _,col=classify_risk(vols_ann[tickers[ri-1]])
            cell.set_text_props(color=col)
    ax6.set_title("H=Historical  C=CAPM Fwd  +  Portfolio β",fontsize=10,color=TXT,fontweight="bold",pad=12)
    return fig

# ─────────────────────────────────────────────────────────────────────
# SECTION 11  ·  FIGURE 3 — RISK PROFILE
# ─────────────────────────────────────────────────────────────────────

def fig_risk_profile(tickers, returns, prices, betas_d,
                     mean_ret_ann, vols_ann, cov_ann, fwd_ret_series):
    sc=plt.cm.plasma(np.linspace(0.12,0.92,len(tickers)))
    fig,axes=plt.subplots(2,3,figsize=(21,12),facecolor=BG)
    fig.suptitle("📋  Portfolio Risk Profile",fontsize=20,color=TXT,fontweight="bold",y=1.005)
    ticks=tickers

    ax=axes[0,0]; x=np.arange(len(ticks))
    ax.bar(x-0.25,[hist_var(returns[t],0.95)*100 for t in ticks],width=0.24,label="VaR 95%",color=WARN,alpha=0.88)
    ax.bar(x,     [hist_cvar(returns[t],0.95)*100 for t in ticks],width=0.24,label="CVaR 95%",color=NEG,alpha=0.88)
    ax.bar(x+0.25,[hist_var(returns[t],0.99)*100 for t in ticks],width=0.24,label="VaR 99%",color=MUT,alpha=0.65)
    ax.set_title("Daily VaR & CVaR (Historical)"); ax.set_ylabel("Daily Loss %")
    ax.set_xticks(x); ax.set_xticklabels(ticks); ax.legend(); ax.yaxis.grid(True); _ax_dark(ax)

    ax=axes[0,1]
    mddv=[max_dd(prices[[t]])*100 for t in ticks]
    bc=[NEG if v<-30 else WARN if v<-15 else POS for v in mddv]
    bars=ax.bar(ticks,mddv,color=bc,alpha=0.85,edgecolor="none")
    for bar,val in zip(bars,mddv):
        ax.text(bar.get_x()+bar.get_width()/2,val-0.8,f"{val:.1f}%",ha="center",va="top",fontsize=9,color=TXT,fontweight="bold")
    ax.set_title("Maximum Peak-to-Trough Drawdown"); ax.set_ylabel("Drawdown %"); ax.yaxis.grid(True); _ax_dark(ax)

    ax=axes[0,2]
    sv=[sortino_s(returns[t]) for t in ticks]
    ax.bar(ticks,sv,color=[POS if v>1 else WARN if v>0 else NEG for v in sv],alpha=0.85,edgecolor="none")
    ax.axhline(1,color=MUT,ls="--",lw=1.2,alpha=0.75,label="Sortino=1")
    ax.set_title("Sortino Ratio (Return / Downside Vol)"); ax.set_ylabel("Sortino"); ax.legend(); ax.yaxis.grid(True); _ax_dark(ax)

    ax=axes[1,0]
    rv=returns[ticks].rolling(60).std()*np.sqrt(TRADING_DAYS)*100
    for i,t in enumerate(ticks): ax.plot(rv.index,rv[t],label=t,lw=1.6,color=sc[i])
    ax.set_title("Rolling 60-Day Annualised Volatility"); ax.set_ylabel("Vol %")
    ax.legend(fontsize=8,ncol=min(len(ticks),4)); ax.yaxis.grid(True); _ax_dark(ax)

    ax=axes[1,1]
    cv=[calmar_s(returns[t],prices[[t]]) for t in ticks]
    bars=ax.bar(ticks,cv,color=[POS if v>1 else WARN if v>0.5 else NEG for v in cv],alpha=0.85,edgecolor="none")
    for bar,val in zip(bars,cv):
        ax.text(bar.get_x()+bar.get_width()/2,max(val+0.02,0.04),f"{val:.2f}",ha="center",va="bottom",fontsize=9,color=TXT,fontweight="bold")
    ax.axhline(1,color=MUT,ls="--",lw=1.2,alpha=0.75,label="Calmar=1")
    ax.set_title("Calmar Ratio (Ann. Return / Max DD)"); ax.set_ylabel("Calmar"); ax.legend(); ax.yaxis.grid(True); _ax_dark(ax)

    ax=axes[1,2]
    ax.bar(np.arange(len(ticks))-0.2,[mean_ret_ann[t]*100 for t in ticks],
           width=0.38,label="Historical",color=ACC,alpha=0.88)
    ax.bar(np.arange(len(ticks))+0.2,[fwd_ret_series[t]*100 for t in ticks],
           width=0.38,label="CAPM Forward",color=POS,alpha=0.88)
    ax.axhline(RF*100,color=MUT,ls="--",lw=1,alpha=0.70,label=f"RF {RF*100:.1f}%")
    ax.set_title("Historical vs CAPM Forward Returns"); ax.set_ylabel("Annual Return %")
    ax.set_xticks(np.arange(len(ticks))); ax.set_xticklabels(ticks); ax.legend(fontsize=8); ax.yaxis.grid(True); _ax_dark(ax)

    plt.tight_layout(pad=2.0); return fig

# ─────────────────────────────────────────────────────────────────────
# SECTION 12  ·  FIGURE 4 — MC FORWARD PROJECTIONS (CAPM-CALIBRATED)
# ─────────────────────────────────────────────────────────────────────

def fig_projections(proj_paths, strategies, initial_value, n_years=PROJ_YEARS):
    t_yr=np.arange(n_years*TRADING_DAYS+1)/TRADING_DAYS
    fig=plt.figure(figsize=(24,18),facecolor=BG)
    fig.suptitle(
        f"📈  CAPM-Calibrated Monte Carlo Projections  ·  "
        f"${initial_value:,.0f} Start  ·  {N_MC_PROJ:,} Paths  ·  {n_years}yr\n"
        f"   Drift anchored to ERP={ERP*100:.1f}% above RF — not raw historical returns",
        fontsize=16,color=TXT,fontweight="bold",y=0.99)
    gs=gridspec.GridSpec(2,3,figure=fig,height_ratios=[2.2,1],
                         hspace=0.38,wspace=0.28,left=0.06,right=0.97,top=0.93,bottom=0.05)

    ax_fan=fig.add_subplot(gs[0,:])
    pname=strategies[0]["name"]; primary=proj_paths[pname]; pcol=strategies[0]["color"]
    ri=np.random.default_rng(0).integers(0,primary.shape[0],min(80,primary.shape[0]))
    for j in ri: ax_fan.plot(t_yr,primary[j],color=pcol,alpha=0.03,lw=0.7,zorder=1)
    pct={p:np.percentile(primary,p,axis=0) for p in [5,10,25,50,75,90,95]}
    ax_fan.fill_between(t_yr,pct[5],pct[95],alpha=0.10,color=pcol,zorder=2)
    ax_fan.fill_between(t_yr,pct[10],pct[90],alpha=0.17,color=pcol,zorder=3)
    ax_fan.fill_between(t_yr,pct[25],pct[75],alpha=0.28,color=pcol,zorder=4,
                        label=f"{pname} 25–75th %ile band")
    ax_fan.plot(t_yr,pct[50],color=pcol,lw=2.8,zorder=5,label=f"{pname} Median")
    for st in strategies[1:]:
        nm=st["name"]
        if nm in proj_paths:
            med=np.median(proj_paths[nm],axis=0)
            ax_fan.plot(t_yr,med,color=st["color"],lw=2,ls="--",alpha=0.85,zorder=5,label=f"{nm} Median")
    ax_fan.plot(t_yr,initial_value*np.exp(RF*t_yr),color=MUT,lw=1.3,ls=":",alpha=0.75,label=f"Risk-Free {RF*100:.1f}%")
    ax_fan.plot(t_yr,initial_value*np.exp(INFLATION_RT*t_yr),color="#a371f7",lw=1.3,ls=":",alpha=0.75,label=f"Inflation {INFLATION_RT*100:.0f}%")
    ax_fan.axhline(initial_value,color=MUT,lw=0.7,alpha=0.35)
    for yr in range(1,n_years+1): ax_fan.axvline(yr,color=MUT,lw=0.35,alpha=0.20)
    for yr in [1,3,5,n_years]:
        if yr<=n_years:
            idx=min(yr*TRADING_DAYS,primary.shape[1]-1); v=pct[50][idx]
            ax_fan.annotate(f"${v:,.0f}",xy=(yr,v),xytext=(5,9),textcoords="offset points",
                            fontsize=8.5,color=pcol,fontweight="bold")
    ax_fan.yaxis.set_major_formatter(plt.FuncFormatter(lambda x,_:f"${x:,.0f}"))
    ax_fan.set_title(f"Portfolio Value Projections  ·  Block Bootstrap + CAPM Drift  ·  Shaded = {pname} confidence",fontsize=13)
    ax_fan.set_xlabel("Years",fontsize=11,color=MUT); ax_fan.set_ylabel("Portfolio Value ($)",fontsize=11,color=MUT)
    ax_fan.legend(ncol=3,fontsize=8.5,loc="upper left"); ax_fan.yaxis.grid(True,alpha=0.18); ax_fan.set_xlim(0,n_years)
    _ax_dark(ax_fan)

    ax_h=fig.add_subplot(gs[1,0])
    for yr,col_h,alp,lbl in [(5,pcol,0.72,"Year 5"),(n_years,ACC,0.50,f"Year {n_years}")]:
        idx=min(yr*TRADING_DAYS,primary.shape[1]-1); vals=primary[:,idx]
        lo,hi=np.percentile(vals,[1,99]); cl=vals[(vals>=lo)&(vals<=hi)]
        ax_h.hist(cl,bins=55,alpha=alp,color=col_h,edgecolor="none",density=True,label=lbl)
        ax_h.axvline(np.median(vals),color=col_h,lw=2,ls="--",alpha=0.9,
                     label=f"Med yr{yr} ${np.median(vals):,.0f}")
    ax_h.axvline(initial_value,color=MUT,lw=1.2,ls="-",alpha=0.6,label="Initial")
    ax_h.set_title(f"Final Value Distribution\n({pname}  Yr 5 & {n_years})",fontsize=11)
    ax_h.set_xlabel("Value ($)"); ax_h.set_ylabel("Density")
    ax_h.xaxis.set_major_formatter(plt.FuncFormatter(lambda x,_:f"${x/1e3:.0f}K"))
    ax_h.legend(fontsize=7.5); ax_h.yaxis.grid(True); _ax_dark(ax_h)

    ax_p=fig.add_subplot(gs[1,1])
    rf_a=initial_value*np.exp(RF*t_yr); inf_a=initial_value*np.exp(INFLATION_RT*t_yr)
    ax_p.plot(t_yr,(primary>initial_value).mean(axis=0)*100,color=POS,lw=2,label="P(profit)")
    ax_p.plot(t_yr,(primary>rf_a[None,:]).mean(axis=0)*100,color=pcol,lw=2,label="P(beat RF)")
    ax_p.plot(t_yr,(primary>inf_a[None,:]).mean(axis=0)*100,color=ACC,lw=2,label="P(beat inflation)")
    ax_p.axhline(50,color=MUT,ls="--",lw=0.9,alpha=0.55)
    ax_p.set_ylim(0,100); ax_p.set_title(f"Probability of Beating Benchmarks\n({pname})",fontsize=11)
    ax_p.set_xlabel("Years"); ax_p.set_ylabel("Probability %")
    ax_p.legend(fontsize=8); ax_p.yaxis.grid(True); _ax_dark(ax_p)

    ax_t=fig.add_subplot(gs[1,2]); ax_t.axis("off")
    hdr=["Strategy","Port β","1yr","5yr",f"{n_years}yr",f"P(▲{n_years}yr)"]
    rows_t=[]
    for st in strategies:
        nm=st["name"]
        if nm not in proj_paths: continue
        p=proj_paths[nm]
        rows_t.append([nm,f"{st['beta']:.2f}",
                        f"${np.median(p[:,min(TRADING_DAYS,p.shape[1]-1)]):,.0f}",
                        f"${np.median(p[:,min(5*TRADING_DAYS,p.shape[1]-1)]):,.0f}",
                        f"${np.median(p[:,-1]):,.0f}",
                        f"{(p[:,-1]>initial_value).mean()*100:.0f}%"])
    tbl=ax_t.table(cellText=rows_t,colLabels=hdr,loc="center",cellLoc="center")
    tbl.auto_set_font_size(False); tbl.set_fontsize(8.5); tbl.scale(1.0,2.35)
    for (ri,ci),cell in tbl.get_celld().items():
        cell.set_facecolor(CARD if ri%2==0 else BG); cell.set_edgecolor("#30363d"); cell.set_linewidth(0.5)
        if ri==0: cell.set_text_props(color=ACC,fontweight="bold")
        elif ci==1: cell.set_text_props(color=WARN,fontweight="bold")
        else: cell.set_text_props(color=TXT)
    ax_t.set_title(f"Projected Median Values (${initial_value:,.0f} invested)\nDrift = CAPM ERP {ERP*100:.1f}%",
                   fontsize=10,color=TXT,fontweight="bold",pad=12)
    return fig

# ─────────────────────────────────────────────────────────────────────
# SECTION 13  ·  FIGURE 5 — ECONOMIC CONDITIONS
# ─────────────────────────────────────────────────────────────────────

def fig_economic(eco, regime, strategies, returns, tickers):
    fig=plt.figure(figsize=(22,14),facecolor=BG)
    fig.suptitle("🌍  Economic Conditions & Portfolio Context",fontsize=20,color=TXT,fontweight="bold",y=0.99)
    gs=gridspec.GridSpec(2,3,figure=fig,hspace=0.42,wspace=0.32,
                         left=0.06,right=0.97,top=0.93,bottom=0.06)

    # 1 — VIX history
    ax=fig.add_subplot(gs[0,0])
    if "VIX" in eco:
        vx=eco["VIX"]["series"].iloc[-252:]
        ax.fill_between(vx.index,vx,alpha=0.25,color=NEG)
        ax.plot(vx.index,vx,color=NEG,lw=1.5)
        vix_now=eco["VIX"]["current"]
        for lvl,col,lbl in [(15,POS,"Low"),(20,WARN,"Moderate"),(30,NEG,"High stress")]:
            ax.axhline(lvl,color=col,ls="--",lw=0.9,alpha=0.7,label=lbl)
        ax.axhline(vix_now,color="white",ls="-",lw=1.5,alpha=0.5)
        ax.set_title(f"CBOE VIX Fear Index  (now {vix_now:.1f})")
        ax.set_ylabel("VIX"); ax.legend(fontsize=7); ax.yaxis.grid(True)
    else:
        ax.text(0.5,0.5,"VIX — no data",ha="center",va="center",transform=ax.transAxes,color=MUT,fontsize=12)
        ax.set_title("CBOE VIX Fear Index")
    _ax_dark(ax)

    # 2 — Yield history (10yr vs 3m)
    ax=fig.add_subplot(gs[0,1])
    if "10Y_YIELD" in eco:
        y10=eco["10Y_YIELD"]["series"].iloc[-252:]
        ax.plot(y10.index,y10,color=WARN,lw=1.8,label="10-Yr Yield")
    if "3M_YIELD" in eco:
        y3=eco["3M_YIELD"]["series"].iloc[-252:]
        ax.plot(y3.index,y3,color=ACC,lw=1.8,label="3-Mo Yield")
    if "10Y_YIELD" in eco and "3M_YIELD" in eco:
        y10s=eco["10Y_YIELD"]["series"]; y3s=eco["3M_YIELD"]["series"]
        idx=y10s.index.intersection(y3s.index); idx=idx[idx>=y10s.index[-252]]
        ax.fill_between(idx,y10s.loc[idx],y3s.loc[idx],
                        where=(y10s.loc[idx]<y3s.loc[idx]),alpha=0.25,color=NEG,label="Inverted")
    ax.set_title("US Treasury Yields  (1yr)"); ax.set_ylabel("Yield %")
    ax.legend(fontsize=7); ax.yaxis.grid(True); _ax_dark(ax)

    # 3 — Sector ETF performance (1yr)
    ax=fig.add_subplot(gs[0,2])
    sector_keys=["XLK","XLF","XLV","XLE","XLI","XLU","XLB","XLRE"]
    sector_names={"XLK":"Tech","XLF":"Fin","XLV":"Health","XLE":"Energy",
                  "XLI":"Ind","XLU":"Util","XLB":"Mater","XLRE":"RE"}
    avail=[(k,eco[k]["chg1y"]*100) for k in sector_keys if k in eco]
    if avail:
        labels=[sector_names.get(k,k) for k,_ in avail]
        vals=[v for _,v in avail]
        bars=ax.bar(labels,vals,color=[POS if v>0 else NEG for v in vals],alpha=0.85,edgecolor="none")
        ax.axhline(0,color=MUT,lw=0.8,alpha=0.5)
        for bar,val in zip(bars,vals):
            ax.text(bar.get_x()+bar.get_width()/2,val+(1 if val>=0 else -3),
                    f"{val:+.0f}%",ha="center",va="bottom" if val>=0 else "top",fontsize=8,color=TXT)
    ax.set_title("GICS Sector ETF Returns  (1yr)"); ax.set_ylabel("Return %"); ax.yaxis.grid(True); _ax_dark(ax)

    # 4 — Regime gauge / summary
    ax=fig.add_subplot(gs[1,0]); ax.axis("off")
    r_color=regime.get("color",MUT); r_label=regime.get("label","UNKNOWN")
    ax.add_patch(mpatches.FancyBboxPatch((0.05,0.55),0.9,0.38,
                  boxstyle="round,pad=0.02",facecolor=r_color,alpha=0.25,
                  edgecolor=r_color,linewidth=2,transform=ax.transAxes))
    ax.text(0.5,0.80,f"Economic Regime",ha="center",va="center",transform=ax.transAxes,
            fontsize=10,color=MUT,style="italic")
    ax.text(0.5,0.70,r_label,ha="center",va="center",transform=ax.transAxes,
            fontsize=16,color=r_color,fontweight="bold")
    for i,bullet in enumerate(regime.get("bullets",[])[:4]):
        y=0.48-i*0.13
        ax.text(0.05,y,"•  "+bullet,ha="left",va="center",transform=ax.transAxes,
                fontsize=8,color=TXT,wrap=True)
    ax.set_title("Economic Regime Assessment",fontsize=11,color=TXT,fontweight="bold",pad=8)

    # 5 — Portfolio beta vs economic cycle
    ax=fig.add_subplot(gs[1,1]); ax.axis("off")
    lines=[
        ("Beta Interpretation","",ACC,True),
        ("< 0.5","Defensive — underperforms in bull markets but",POS,False),
        ("","resilient in downturns (utilities, staples)","",False),
        ("0.5–0.85","Moderate — less volatile than market",ACC,False),
        ("0.85–1.15","Market-like — tracks index closely",MUT,False),
        ("1.15–1.5","Aggressive — amplifies market moves",WARN,False),
        ("> 1.5","Very Aggressive — high reward/risk",NEG,False),
        ("","","",False),
        ("Your Portfolios:","",TXT,True),
    ]
    for st in strategies:
        lines.append((f"  {st['name']}: β={st['beta']:.2f}",
                      f"({classify_beta(st['beta'])})",st["color"],False))
    y=0.97
    for lbl,extra,col,bold in lines:
        ax.text(0.05,y,lbl,ha="left",va="top",transform=ax.transAxes,
                fontsize=9 if not bold else 10,color=col if col else TXT,
                fontweight="bold" if bold else "normal")
        if extra: ax.text(0.45,y,extra,ha="left",va="top",transform=ax.transAxes,fontsize=8.5,color=MUT)
        y-=0.10
    ax.set_title("Portfolio Beta & Economic Sensitivity",fontsize=11,color=TXT,fontweight="bold",pad=8)

    # 6 — Portfolio vs SPX (rolling 1yr correlation)
    ax=fig.add_subplot(gs[1,2])
    if "SPX" in eco:
        spx=eco["SPX"]["series"]; spx_ret=spx.pct_change().dropna()
        for i,t in enumerate(tickers[:5]):
            if t in returns.columns:
                s=returns[t]; common=s.index.intersection(spx_ret.index)
                if len(common)>60:
                    roll_c=s.loc[common].rolling(60).corr(spx_ret.loc[common])
                    sc_c=plt.cm.plasma(np.linspace(0.12,0.92,len(tickers)))
                    ax.plot(roll_c.index,roll_c,label=t,lw=1.5,color=sc_c[i])
        ax.axhline(1,color=WARN,ls="--",lw=0.8,alpha=0.6,label="Perfect correlation")
        ax.axhline(0,color=MUT, ls="-", lw=0.5,alpha=0.4)
        ax.set_title("Rolling 60-Day Correlation vs S&P 500"); ax.set_ylabel("Correlation")
        ax.legend(fontsize=7,ncol=2); ax.yaxis.grid(True)
    else:
        ax.text(0.5,0.5,"S&P 500 data unavailable",ha="center",va="center",transform=ax.transAxes,color=MUT)
        ax.set_title("Rolling Correlation vs S&P 500")
    _ax_dark(ax)
    plt.tight_layout(pad=2.0); return fig

# ─────────────────────────────────────────────────────────────────────
# SECTION 14  ·  FIGURE 6 — 3D MARKET RISK SURFACE
# ─────────────────────────────────────────────────────────────────────

def fig_3d_risk(proj_paths, strategies, initial_value, fwd_ret_s, cov_ann, beta_vec,
                n_years=PROJ_YEARS):
    fig=plt.figure(figsize=(24,12),facecolor=BG)
    fig.suptitle("🧊  3D Market Risk Surface",fontsize=20,color=TXT,fontweight="bold",y=1.01)

    # ── LEFT: 3D probability distribution surface ─────────────────────
    ax1=fig.add_subplot(121,projection="3d"); _ax_dark3d(ax1)
    pname=strategies[0]["name"]; primary=proj_paths[pname]
    years=np.arange(1,n_years+1); n_bins=35

    all_vals=np.concatenate([primary[:,min(yr*TRADING_DAYS,primary.shape[1]-1)] for yr in years])
    vlo,vhi=np.percentile(all_vals,[1,99])
    edges=np.linspace(vlo,vhi,n_bins+1); centers=(edges[:-1]+edges[1:])/2

    Z=np.zeros((len(years),n_bins))
    for i,yr in enumerate(years):
        idx=min(yr*TRADING_DAYS,primary.shape[1]-1)
        hist,_=np.histogram(primary[:,idx],bins=edges,density=True); Z[i]=hist

    Xg=np.tile(years[:,None],(1,n_bins)); Yg=np.tile(centers[None,:],(len(years),1))
    norm=plt.Normalize(Z.min(),Z.max())
    surf=ax1.plot_surface(Xg,Yg,Z,facecolors=plt.cm.plasma(norm(Z)),alpha=0.85,shade=True,linewidth=0)
    sm=plt.cm.ScalarMappable(cmap="plasma",norm=norm); sm.set_array([])
    cb=fig.colorbar(sm,ax=ax1,shrink=0.5,aspect=15,pad=0.1)
    cb.set_label("Probability Density",color=MUT,fontsize=8)
    plt.setp(cb.ax.yaxis.get_ticklabels(),color=MUT,fontsize=7)
    ax1.set_xlabel("Year",fontsize=9,color=TXT,labelpad=8)
    ax1.set_ylabel(f"Portfolio Value ($)",fontsize=9,color=TXT,labelpad=12)
    ax1.set_zlabel("Prob. Density",fontsize=9,color=TXT,labelpad=8)
    ax1.set_title(f"{pname}\nOutcome Probability Distribution Over Time",fontsize=11,color=TXT,pad=15)
    fmt=plt.FuncFormatter(lambda x,_:f"${x/1e3:.0f}K")
    ax1.yaxis.set_major_formatter(fmt); ax1.xaxis.set_major_locator(plt.MaxNLocator(integer=True))
    ax1.view_init(elev=28,azim=-55)

    # Median spine
    med_vals=[np.median(primary[:,min(yr*TRADING_DAYS,primary.shape[1]-1)]) for yr in years]
    med_density=[float(np.histogram([mv],bins=edges,density=True)[0][0])
                 if edges[0]<=mv<=edges[-1] else 0 for yr,mv in zip(years,med_vals)]
    ax1.plot(years,med_vals,med_density,color=WARN,lw=2.5,zorder=10,label="Median path")
    ax1.legend(loc="upper left",fontsize=8,framealpha=0.3)

    # ── RIGHT: 3D Risk-Return-Beta cloud ──────────────────────────────
    ax2=fig.add_subplot(122,projection="3d"); _ax_dark3d(ax2)
    k=len(fwd_ret_s); rng=np.random.default_rng(7); n_mc=6000
    ws=rng.dirichlet(np.ones(k),size=n_mc)
    rets=ws@fwd_ret_s.values; vols=np.sqrt(np.einsum("ni,ij,nj->n",ws,cov_ann.values,ws))
    bts=ws@beta_vec; srs=(rets-RF)/np.where(vols>0,vols,1e-9)
    sc2=ax2.scatter(vols*100,rets*100,bts,c=srs,cmap="plasma",alpha=0.25,s=3,zorder=2)
    cb2=fig.colorbar(sc2,ax=ax2,shrink=0.5,aspect=15,pad=0.1)
    cb2.set_label("Sharpe Ratio",color=MUT,fontsize=8)
    plt.setp(cb2.ax.yaxis.get_ticklabels(),color=MUT,fontsize=7)
    for st in strategies:
        r,v,sr=port_metrics(st["raw_w"],0,fwd_ret_s.values,cov_ann.values)
        pb=st["beta"]
        ax2.scatter([v*100],[r*100],[pb],marker=st["marker"],s=st.get("ms",200)*1.5,
                    color=st["color"],zorder=10,edgecolors="white",linewidths=0.8,
                    label=f"{st['name']} β={pb:.2f}")
    ax2.set_xlabel("Ann. Volatility (%)",fontsize=9,color=TXT,labelpad=8)
    ax2.set_ylabel("CAPM Return (%)",fontsize=9,color=TXT,labelpad=8)
    ax2.set_zlabel("Portfolio Beta",fontsize=9,color=TXT,labelpad=8)
    ax2.set_title("3D Risk Space: Vol × Return × Beta\n(colour = Sharpe Ratio)",fontsize=11,color=TXT,pad=15)
    ax2.legend(loc="upper left",fontsize=8,framealpha=0.3); ax2.view_init(elev=22,azim=40)
    plt.tight_layout(pad=2.0); return fig

# ─────────────────────────────────────────────────────────────────────
# SECTION 15  ·  FIGURE 7 — DIVERSIFICATION RECOMMENDATIONS
# ─────────────────────────────────────────────────────────────────────

def fig_diversification(tickers, weights_main, sector_exp, recs, corr_df, regime):
    fig=plt.figure(figsize=(24,16),facecolor=BG)
    fig.suptitle("🔀  Diversification Analysis & Recommendations",fontsize=20,color=TXT,fontweight="bold",y=0.99)
    gs=gridspec.GridSpec(2,3,figure=fig,hspace=0.45,wspace=0.30,
                         left=0.05,right=0.97,top=0.93,bottom=0.05)

    # P1 — Sector pie
    ax=fig.add_subplot(gs[0,0]); ax.set_facecolor(BG)
    if sector_exp:
        labels=list(sector_exp.keys()); sizes=list(sector_exp.values())
        colors_p=plt.cm.plasma(np.linspace(0.1,0.9,len(labels)))
        wedges,texts,autotexts=ax.pie(sizes,labels=labels,colors=colors_p,autopct="%1.0f%%",
                                       startangle=90,pctdistance=0.82,
                                       wedgeprops={"edgecolor":BG,"linewidth":1.2})
        for t in texts: t.set_color(TXT); t.set_fontsize(8)
        for at in autotexts: at.set_color(BG); at.set_fontsize(7.5); at.set_fontweight("bold")
    ax.set_title("Current Portfolio\nSector Exposure",fontsize=11,color=TXT,fontweight="bold",pad=8)

    # P2 — Correlation heatmap (small)
    ax=fig.add_subplot(gs[0,1]); _ax_dark(ax)
    if len(tickers)>1:
        corr_sub=corr_df.loc[tickers,tickers]
        sns.heatmap(corr_sub,annot=True,fmt=".2f",cmap=sns.diverging_palette(12,220,sep=10,n=256,as_cmap=True),
                    vmin=-1,vmax=1,center=0,ax=ax,square=True,linewidths=0.5,linecolor="#30363d",
                    annot_kws={"size":max(7,11-len(tickers)),"weight":"bold"},
                    cbar_kws={"shrink":0.7,"aspect":18,"pad":0.02})
    ax.set_title("Asset Correlation  (high = less diversification)",fontsize=11,color=TXT,fontweight="bold",pad=8)
    ax.tick_params(axis="x",rotation=45,labelsize=9,colors=TXT)
    ax.tick_params(axis="y",rotation=0, labelsize=9,colors=TXT)

    # P3 — HHI concentration bar
    ax=fig.add_subplot(gs[0,2]); _ax_dark(ax)
    h=hhi(list(sector_exp.values())); n_s=len(sector_exp)
    perfect=1.0/n_s if n_s else 1
    ax.barh(["Portfolio HHI"],[h],color=NEG if h>0.35 else WARN if h>0.20 else POS,alpha=0.88,edgecolor="none")
    ax.barh(["Perfect Equal"],[perfect],color=POS,alpha=0.5,edgecolor="none")
    ax.axvline(0.25,color=WARN,ls="--",lw=1.2,alpha=0.8,label="Moderate concentration (0.25)")
    ax.axvline(0.40,color=NEG, ls="--",lw=1.2,alpha=0.8,label="High concentration (0.40)")
    ax.set_xlim(0,1); ax.set_title("Sector Concentration\n(HHI — lower = better diversified)",fontsize=11,color=TXT,fontweight="bold",pad=8)
    ax.set_xlabel("HHI Score (0=perfect diversification, 1=monopoly)"); ax.legend(fontsize=8); ax.xaxis.grid(True)

    # P4–P6 — Recommendation cards
    for card_i in range(3):
        ax=fig.add_subplot(gs[1,card_i]); ax.axis("off")
        batch=recs[card_i*3:(card_i+1)*3]
        y=0.97
        for j,rec in enumerate(batch):
            rtype=rec["type"]; tcol=NEG if rtype=="OVER-CONCENTRATED" else POS
            ax.add_patch(mpatches.FancyBboxPatch((0.02,y-0.30),0.96,0.30,
                         boxstyle="round,pad=0.01",facecolor=CARD,alpha=0.9,
                         edgecolor=tcol,linewidth=1.0,transform=ax.transAxes))
            ax.text(0.05,y-0.03,f"[{rtype}]  {rec['sector']}  →  {rec['ticker']}",
                    ha="left",va="top",transform=ax.transAxes,
                    fontsize=9,color=tcol,fontweight="bold")
            short_desc=rec["desc"][:80]+"…" if len(rec["desc"])>80 else rec["desc"]
            ax.text(0.05,y-0.10,short_desc,ha="left",va="top",transform=ax.transAxes,
                    fontsize=7.5,color=TXT)
            why_short=rec["why"][:90]+"…" if len(rec["why"])>90 else rec["why"]
            ax.text(0.05,y-0.19,f"Why: {why_short}",ha="left",va="top",transform=ax.transAxes,
                    fontsize=7,color=MUT,style="italic")
            if rec["avg_corr"]!="N/A":
                ax.text(0.78,y-0.03,f"Avg corr: {rec['avg_corr']}",ha="left",va="top",
                        transform=ax.transAxes,fontsize=7.5,color=ACC)
            y-=0.33
        ax.set_title(f"Top Recommendations ({card_i*3+1}–{min(card_i*3+3,len(recs))})",
                     fontsize=11,color=TXT,fontweight="bold",pad=8)
    return fig

# ─────────────────────────────────────────────────────────────────────
# SECTION 16  ·  CONSOLE REPORT
# ─────────────────────────────────────────────────────────────────────

def print_report(tickers, returns, prices, betas_d, mean_ret_ann, vols_ann,
                 fwd_ret_s, cov_ann, strategies, proj_paths, initial_value,
                 regime, recs, sector_exp):
    W=80; S="═"*W; D="─"*W

    print(f"\n{S}\n  {'INDIVIDUAL STOCK METRICS':^{W-4}}\n{S}")
    print(f"  {'Ticker':<8} {'Hist%':>8} {'CAPM%':>8} {'Vol%':>8} {'Sharpe':>7} {'Beta':>7} {'MaxDD%':>8} {'Risk':>9}  Beta Profile")
    print(D)
    for t in tickers:
        r=mean_ret_ann[t]; v=vols_ann[t]; beta=betas_d.get(t,np.nan)
        fr=fwd_ret_s[t]; mdd=max_dd(prices[[t]])*100; rl,_=classify_risk(v)
        print(f"  {t:<8} {r*100:>+7.2f}% {fr*100:>+7.2f}% {v*100:>7.2f}% "
              f"{sharpe(r,v):>7.2f} {beta:>7.2f} {mdd:>7.2f}% {rl:>9}  {classify_beta(beta)}")

    print(f"\n{S}\n  {'TAIL RISK  (daily, historical)':^{W-4}}\n{S}")
    print(f"  {'Ticker':<8} {'VaR95%':>8} {'CVaR95%':>9} {'VaR99%':>8} {'Sortino':>9} {'Calmar':>9}")
    print(D)
    for t in tickers:
        print(f"  {t:<8} {hist_var(returns[t],0.95)*100:>7.2f}% "
              f"{hist_cvar(returns[t],0.95)*100:>8.2f}% "
              f"{hist_var(returns[t],0.99)*100:>7.2f}% "
              f"{sortino_s(returns[t]):>9.2f} "
              f"{calmar_s(returns[t],prices[[t]]):>9.2f}")

    print(f"\n{S}\n  {'OPTIMAL PORTFOLIOS  (CAPM-calibrated, incl. Cash)':^{W-4}}\n{S}")
    print(f"  {'Strategy':<22} {'Port β':>7} {'FwdRet%':>8} {'Vol%':>8} {'Sharpe':>8} {'Cash%':>7} {'Risk':>10}")
    print(D)
    for st in strategies:
        r,v,sr=port_metrics(st["sw"],st["cw"],fwd_ret_s.values,cov_ann.values); rl,_=classify_risk(v)
        print(f"  {st['name']:<22} {st['beta']:>7.2f} {r*100:>+7.2f}% {v*100:>7.2f}% {sr:>8.2f} {st['cw']*100:>6.1f}% {rl:>10}")

    print(f"\n{S}\n  {'RECOMMENDED WEIGHTINGS':^{W-4}}\n{S}")
    for st in strategies:
        r,v,sr=port_metrics(st["sw"],st["cw"],fwd_ret_s.values,cov_ann.values); rl,_=classify_risk(v)
        print(f"\n  ▶  {st['name']}  │  FwdRet:{r*100:+.2f}%  Vol:{v*100:.2f}%  SR:{sr:.2f}  β:{st['beta']:.2f}  Risk:{rl}")
        for t,w in zip(tickers,st["sw"]):
            bar="█"*int(w*44); print(f"     {t:<8} {bar:<44} {w*100:>5.1f}%")
        cb="█"*int(st["cw"]*44); print(f"     {'CASH':<8} {cb:<44} {st['cw']*100:>5.1f}%")

    print(f"\n{S}\n  {'MC FORWARD PROJECTIONS  (CAPM ERP={:.1f}%)'.format(ERP*100):^{W-4}}\n{S}")
    print(f"  {'Strategy':<22} {'β':>5} {'1yr':>10} {'5yr':>10} {f'{PROJ_YEARS}yr':>10} {f'P(▲{PROJ_YEARS}yr)':>12}")
    print(D)
    for st in strategies:
        nm=st["name"]
        if nm not in proj_paths: continue
        p=proj_paths[nm]
        print(f"  {nm:<22} {st['beta']:>5.2f} "
              f"${np.median(p[:,min(TRADING_DAYS,p.shape[1]-1)]):>9,.0f} "
              f"${np.median(p[:,min(5*TRADING_DAYS,p.shape[1]-1)]):>9,.0f} "
              f"${np.median(p[:,-1]):>9,.0f} "
              f"{(p[:,-1]>initial_value).mean()*100:>11.0f}%")

    print(f"\n{S}\n  {'ECONOMIC REGIME':^{W-4}}\n{S}")
    print(f"  Regime: {regime.get('label','UNKNOWN')}")
    for b in regime.get("bullets",[]): print(f"  • {b}")

    print(f"\n{S}\n  {'DIVERSIFICATION RECOMMENDATIONS':^{W-4}}\n{S}")
    for i,rec in enumerate(recs,1):
        print(f"\n  [{i}] {rec['type']:25}  Sector: {rec['sector']}")
        print(f"      Add: {rec['ticker']}")
        print(f"      {rec['desc']}")
        print(f"      Why: {rec['why']}")
        if rec['avg_corr']!='N/A':
            print(f"      Average correlation with current portfolio: {rec['avg_corr']}")
        if rec.get("all_recs") and len(rec["all_recs"])>1:
            print(f"      Alternatives: "+", ".join(f"{t} ({d[:40]}…)" for t,d in rec["all_recs"][1:3]))

    print(f"\n{D}")
    print(f"  Model notes:"); print(f"    CAPM ERP assumed  : {ERP*100:.1f}%  |  RF: {RF*100:.2f}%  |  Inflation: {INFLATION_RT*100:.0f}%")
    print(f"    Projection method : CAPM-drift block-bootstrap (block=20 days)")
    print(f"    Weight bounds     : {MIN_W*100:.0f}%–{MAX_W*100:.0f}% per asset")
    print(f"\n  ⚠️  Past performance ≠ future results. Educational use only.\n{S}\n")

# ─────────────────────────────────────────────────────────────────────
# SECTION 17  ·  MAIN
# ─────────────────────────────────────────────────────────────────────

def main():
    try:
        import matplotlib; matplotlib.use("TkAgg")
    except Exception:
        try: import matplotlib; matplotlib.use("Qt5Agg")
        except Exception: pass

    setup_style()
    tickers, mode, user_sw, user_cw, initial_value = get_user_inputs()

    # ── Fetch stock prices ────────────────────────────────────────────
    try:   prices, market = fetch_prices(tickers)
    except Exception as e: print(f"\n  ❌  Price download failed: {e}\n"); sys.exit(1)
    tickers = validate(tickers, prices)
    if len(tickers)<2: print("\n  ❌  Need ≥2 valid tickers.\n"); sys.exit(1)
    prices  = prices[tickers]
    print(f"  ✅  {len(prices)} trading days  "
          f"({prices.index[0].date()} → {prices.index[-1].date()})")

    # ── Compute stock metrics ─────────────────────────────────────────
    returns       = prices.pct_change().dropna()
    mkt_rets      = market.pct_change().dropna() if market is not None else None
    mean_ret_ann  = ann_ret(returns)      # historical (for display only)
    vols_ann      = ann_vol(returns)      # historical vol (used in optim & MC sigma)
    cov_ann       = returns.cov() * TRADING_DAYS
    betas_d       = compute_betas(returns, mkt_rets) if mkt_rets is not None else {t:np.nan for t in tickers}
    beta_vec      = np.array([betas_d.get(t,0.0) for t in tickers])

    # ── CAPM forward returns (used in optimisation AND projections) ───
    fwd_ret_s = pd.Series({t: capm_forward(betas_d.get(t,1.0)) for t in tickers})
    print(f"  📐  CAPM forward returns: "+" | ".join(f"{t}={fwd_ret_s[t]*100:.1f}%" for t in tickers))

    # ── Optimise portfolios ───────────────────────────────────────────
    print("  ⚙️   Optimising portfolios (Max Sharpe, Min Vol, Risk Parity) …")
    w_sr_r  = opt_weights(fwd_ret_s.values, cov_ann.values, "sharpe")
    w_mv_r  = opt_weights(fwd_ret_s.values, cov_ann.values, "min_vol")
    w_rp_r  = risk_parity(cov_ann.values)

    sc_cash = _auto_cash(cov_ann.values, w_sr_r)
    w_sr_s, w_sr_c = apply_cash(w_sr_r, sc_cash)
    w_mv_s, w_mv_c = apply_cash(w_mv_r, max(0.03, sc_cash*0.40))
    w_rp_s, w_rp_c = apply_cash(w_rp_r, sc_cash*0.65)

    strategies = [
        {"name":"Max Sharpe",    "sw":w_sr_s,"cw":w_sr_c,"raw_w":w_sr_r,
         "beta":port_beta(w_sr_s,beta_vec),"color":WARN,"marker":"*","ms":380},
        {"name":"Min Volatility","sw":w_mv_s,"cw":w_mv_c,"raw_w":w_mv_r,
         "beta":port_beta(w_mv_s,beta_vec),"color":POS, "marker":"D","ms":150},
        {"name":"Risk Parity",   "sw":w_rp_s,"cw":w_rp_c,"raw_w":w_rp_r,
         "beta":port_beta(w_rp_s,beta_vec),"color":RP,  "marker":"P","ms":190},
    ]
    if mode in (2,3) and user_sw is not None:
        usw = user_sw[:len(tickers)] if len(user_sw)>len(tickers) else user_sw
        if len(usw)<len(tickers): usw=np.append(usw,np.zeros(len(tickers)-len(usw)))
        strategies.append({"name":"Your Portfolio","sw":usw,"cw":user_cw,
                            "raw_w":usw/max(usw.sum(),1e-9),"beta":port_beta(usw,beta_vec),
                            "color":USR,"marker":"H","ms":200})

    # ── Economic data ─────────────────────────────────────────────────
    eco    = fetch_economic_data()
    regime = analyse_regime(eco)
    print(f"  📊  Economic regime: {regime['label']}")

    # ── Diversification analysis ──────────────────────────────────────
    primary_sw = strategies[0]["sw"]
    sec_exp    = sector_exposure(tickers, primary_sw / max(primary_sw.sum(),1e-9))
    corr_df    = returns.corr()
    recs       = build_recommendations(tickers, sec_exp, regime, corr_df)
    hhi_score  = hhi(list(sec_exp.values()))
    div_score  = diversification_score(returns)
    print(f"  🔀  HHI concentration score: {hhi_score:.3f}  (≤0.18=well diversified)  |  Div score: {div_score:.2f}")

    # ── Construction MC cloud ─────────────────────────────────────────
    print(f"  🎲  Building efficient frontier cloud ({N_MC_CLOUD:,} portfolios) …")
    mc_res = mc_cloud(fwd_ret_s.values, cov_ann.values)

    # ── Forward projection simulations ───────────────────────────────
    print(f"  📈  Running CAPM-calibrated forward simulations "
          f"({N_MC_PROJ:,} paths × {PROJ_YEARS}yr) …")
    proj_paths = {}
    for st in strategies:
        proj_paths[st["name"]] = simulate_capm_bootstrap(
            st["sw"], st["cw"], returns, fwd_ret_s, cov_ann,
            n_years=PROJ_YEARS, n_paths=N_MC_PROJ,
            initial=initial_value, seed=42)

    # ── Console report ────────────────────────────────────────────────
    print_report(tickers, returns, prices, betas_d, mean_ret_ann, vols_ann,
                 fwd_ret_s, cov_ann, strategies, proj_paths, initial_value,
                 regime, recs, sec_exp)

    # ── Render all 7 figures ──────────────────────────────────────────
    print("  📊  Rendering 7 chart windows … (may take a moment)\n")
    figs = [
        ("Correlation Heatmap",        fig_heatmap(returns, tickers)),
        ("Portfolio Dashboard",         fig_dashboard(tickers, returns, prices, betas_d,
                                                       mean_ret_ann, vols_ann, mc_res,
                                                       cov_ann, strategies, fwd_ret_s)),
        ("Risk Profile",               fig_risk_profile(tickers, returns, prices, betas_d,
                                                         mean_ret_ann, vols_ann, cov_ann, fwd_ret_s)),
        ("MC Projections (CAPM)",      fig_projections(proj_paths, strategies, initial_value)),
        ("Economic Conditions",         fig_economic(eco, regime, strategies, returns, tickers)),
        ("3D Market Risk Surface",      fig_3d_risk(proj_paths, strategies, initial_value,
                                                     fwd_ret_s, cov_ann, beta_vec)),
        ("Diversification Recs",        fig_diversification(tickers, primary_sw, sec_exp,
                                                              recs, corr_df, regime)),
    ]
    for title, fig in figs:
        try: fig.canvas.manager.set_window_title(title)
        except Exception: pass

    print("  Save all charts as PNG? [y / N]: ", end="", flush=True)
    try:    ans=input().strip().lower()
    except EOFError: ans="n"
    if ans=="y":
        for i,(title,fig) in enumerate(figs,1):
            fname=f"portfolio_{i}_{title.replace(' ','_').replace('/','')}.png"
            fig.savefig(fname,dpi=150,bbox_inches="tight",facecolor=BG)
            print(f"  💾  {fname}")

    plt.show()
    print("  ✅  Analysis complete. Close chart windows to exit.\n")

if __name__ == "__main__":
    main()



KeyboardInterrupt: Interrupted by user